# NMI and ARI — from scratch
The two clustering quality measures you'll use most in your thesis. We build both from first principles with numpy, then verify against scikit-learn.

You'll see:

1. The setting: two partitions of N points
2. **Rand Index** — counting pairs of points
3. Why a raw Rand Index is misleading
4. **Adjusted Rand Index (ARI)** — chance-corrected
5. **Entropy** — the unit of information
6. **Mutual Information** — how much one partition tells you about the other
7. **Normalized Mutual Information (NMI)**
8. Sanity checks on edge cases (perfect, random, permuted, single-cluster)
9. How you'll actually use them in your thesis

Both metrics live in `sklearn.metrics`:
`adjusted_rand_score(true, pred)` and `normalized_mutual_info_score(true, pred)`.


## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import comb        # n choose k — pair counting helper

np.set_printoptions(precision=3, suppress=True)


## 1. The setting

You have **N points**. Two label arrays of length N:

- `y_true` — the true class of each point.
- `y_pred` — the cluster k-means (or your encoder + k-means) put each point into.

A **partition** doesn't care about the *names* of the clusters — only about *which points are grouped together*. So `[0,0,1,1]` and `[5,5,9,9]` describe the same partition. Both metrics respect this — they are invariant under relabelling.

A small example we'll carry through the notebook:


In [ ]:
y_true = np.array([0, 0, 0, 1, 1, 1])     # 6 points, 3 in class 0, 3 in class 1
y_pred = np.array([0, 0, 1, 1, 1, 1])     # the predicted clustering — one point is misplaced

N = len(y_true)
print("N =", N)
print("y_true:", y_true)
print("y_pred:", y_pred)


Let's visualise it. The points are placed on a line; colour = the label assigned by each partition.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 2.5), sharex=True)
xs = np.arange(N)

axes[0].scatter(xs, np.zeros(N), c=y_true, cmap="tab10", s=200, edgecolor="black")
axes[0].set_title("True partition (y_true)")
axes[0].set_yticks([])

axes[1].scatter(xs, np.zeros(N), c=y_pred, cmap="tab10", s=200, edgecolor="black")
axes[1].set_title("Predicted partition (y_pred)")
axes[1].set_yticks([])
axes[1].set_xticks(xs)
plt.tight_layout()
plt.show()


**Notice** point 2: true class 0 (left group), but predicted cluster 1 (right group). That's the disagreement we want our metrics to penalise.

## 2. Rand Index — counting pairs

The **Rand Index** asks, for every pair of points, the same yes/no question:

> *Do `y_true` and `y_pred` agree on whether these two points belong together?*

For each unordered pair $(i, j)$ there are four possibilities:

| | true: same cluster | true: different cluster |
|---|---|---|
| **pred: same cluster** | `a` (agree — together) | `b` (disagree) |
| **pred: different cluster** | `c` (disagree) | `d` (agree — apart) |

So `a + b + c + d` = total pairs = $\binom{N}{2}$.

$$ \text{RI} = \frac{a + d}{a + b + c + d} = \frac{\text{agreements}}{\text{total pairs}} $$

A value of 1 means the two partitions agree on every single pair. 0 means they disagree on every pair.


In [ ]:
def rand_index_from_scratch(y_true, y_pred):
    n = len(y_true)
    a = b = c = d = 0
    for i in range(n):
        for j in range(i + 1, n):
            same_true = y_true[i] == y_true[j]
            same_pred = y_pred[i] == y_pred[j]
            if   same_true and  same_pred: a += 1
            elif same_true and not same_pred: c += 1
            elif not same_true and same_pred: b += 1
            else: d += 1
    total = a + b + c + d
    return (a + d) / total, (a, b, c, d)


ri, (a, b, c, d) = rand_index_from_scratch(y_true, y_pred)
print(f"a (same/same)   = {a}")
print(f"b (diff/same)   = {b}")
print(f"c (same/diff)   = {c}")
print(f"d (diff/diff)   = {d}")
print(f"Total pairs     = {a+b+c+d}  (should equal C(N,2) = {comb(N, 2)})")
print(f"Rand Index      = {ri:.3f}")


So our toy example has RI ≈ **0.667** — the two partitions agree on 10 of 15 pairs.

## 3. Why Rand Index alone is misleading

Even **random** clusterings can score a surprisingly high RI, because most pairs of points belong to *different* clusters (contributing to `d`). Let's verify.


In [ ]:
rng = np.random.default_rng(seed=42)

# Take a fixed y_true with 3 classes, 60 points
N_big = 60
y_true_big = np.repeat([0, 1, 2], 20)

# Generate 5000 random clusterings into 3 clusters and average the Rand Index
ri_random = []
for _ in range(5000):
    y_pred_random = rng.integers(0, 3, size=N_big)
    ri_random.append(rand_index_from_scratch(y_true_big, y_pred_random)[0])

print(f"Average RI of random clusterings: {np.mean(ri_random):.3f}")
print(f"(A clustering that's chance-level should be 'around 0', but RI ≈ 0.55 — misleading.)")


**Conclusion:** raw Rand Index doesn't give 0 for random clusterings. We need to subtract off the chance baseline. That's what ARI does.

## 4. Adjusted Rand Index (ARI)

Idea: rescale the Rand Index so that

- **random** clusterings score **≈ 0**,
- **perfect** clusterings score **= 1**,
- values can go slightly negative for worse-than-random.

The standard (Hubert–Arabie) formula uses the **contingency table** $n_{ij}$ = number of points in predicted cluster $i$ **and** true class $j$.

Let $a_i = \sum_j n_{ij}$ (predicted cluster sizes) and $b_j = \sum_i n_{ij}$ (true class sizes). Then

$$
\text{ARI} =
\frac{\sum_{ij} \binom{n_{ij}}{2} \;-\; \frac{\sum_i \binom{a_i}{2} \sum_j \binom{b_j}{2}}{\binom{N}{2}}}
     {\frac{1}{2}\!\left(\sum_i \binom{a_i}{2} + \sum_j \binom{b_j}{2}\right) - \frac{\sum_i \binom{a_i}{2} \sum_j \binom{b_j}{2}}{\binom{N}{2}}}
$$

Looks ugly. The structure is just: **(Index − ExpectedIndex) / (MaxIndex − ExpectedIndex)** — exactly what „adjusted" means.


In [ ]:
def contingency_table(y_true, y_pred):
    """Build the contingency matrix n_ij."""
    classes = np.unique(y_true)
    clusters = np.unique(y_pred)
    M = np.zeros((len(clusters), len(classes)), dtype=int)
    for i, c in enumerate(clusters):
        for j, k in enumerate(classes):
            M[i, j] = np.sum((y_pred == c) & (y_true == k))
    return M


M = contingency_table(y_true, y_pred)
print("Contingency table (rows = predicted cluster, cols = true class):")
print(M)
print("Row sums (predicted cluster sizes):", M.sum(axis=1))
print("Col sums (true class sizes):       ", M.sum(axis=0))


In [ ]:
def ari_from_scratch(y_true, y_pred):
    M = contingency_table(y_true, y_pred)
    N = M.sum()
    sum_nij = sum(comb(n, 2) for n in M.flatten())
    sum_ai  = sum(comb(int(a), 2) for a in M.sum(axis=1))
    sum_bj  = sum(comb(int(b), 2) for b in M.sum(axis=0))
    expected = sum_ai * sum_bj / comb(N, 2)
    max_ix   = 0.5 * (sum_ai + sum_bj)
    return (sum_nij - expected) / (max_ix - expected)


ari = ari_from_scratch(y_true, y_pred)
print(f"ARI from scratch : {ari:.3f}")

# Verify against sklearn
from sklearn.metrics import adjusted_rand_score
print(f"ARI from sklearn : {adjusted_rand_score(y_true, y_pred):.3f}")


**The two numbers match.** You now know exactly what ARI computes.

A useful intuition: ARI = 1 − (a corrected fraction of disagreements). For our toy example: ARI ≈ 0.24 — better than chance, but the misplaced point hurts.


## 5. Entropy — the foundation of NMI

Now we switch to the **information-theoretic** view.

**Entropy** of a label array $L$ measures how unpredictable the labels are:

$$ H(L) = -\sum_k p_k \log p_k $$

where $p_k$ is the fraction of points with label $k$.

- High entropy = labels are spread out evenly (hard to guess).
- Low entropy = labels are concentrated on a few values.
- $H = 0$ when all points have the same label (no uncertainty).


In [ ]:
def entropy(labels):
    """Shannon entropy (in nats — natural log)."""
    _, counts = np.unique(labels, return_counts=True)
    p = counts / counts.sum()
    return -np.sum(p * np.log(p))


print(f"H(y_true) = {entropy(y_true):.3f}")
print(f"H(y_pred) = {entropy(y_pred):.3f}")

# Edge cases
print()
print(f"H of all-same labels: {entropy(np.zeros(10)):.3f}      (no uncertainty)")
print(f"H of perfectly balanced 2-class: {entropy([0]*5+[1]*5):.3f}  (= log(2))")
print(f"log(2) = {np.log(2):.3f}")


## 6. Mutual Information

If $H(L)$ measures the uncertainty in one labelling, **mutual information** $I(L_1; L_2)$ measures how much knowing $L_2$ **reduces** uncertainty about $L_1$.

$$ I(L_1; L_2) \;=\; H(L_1) + H(L_2) - H(L_1, L_2) $$

where $H(L_1, L_2)$ is the joint entropy: entropy of the (true, pred) pairs.

- $I = 0$ when the two labellings are **independent** (knowing one tells you nothing about the other).
- $I$ is large when the two labellings are **strongly correlated**.


In [ ]:
def joint_entropy(y_true, y_pred):
    M = contingency_table(y_true, y_pred).flatten()
    p = M / M.sum()
    p = p[p > 0]   # skip zeros — 0 * log(0) := 0
    return -np.sum(p * np.log(p))


def mutual_information(y_true, y_pred):
    return entropy(y_true) + entropy(y_pred) - joint_entropy(y_true, y_pred)


I = mutual_information(y_true, y_pred)
print(f"H(y_true)         = {entropy(y_true):.3f}")
print(f"H(y_pred)         = {entropy(y_pred):.3f}")
print(f"H(y_true, y_pred) = {joint_entropy(y_true, y_pred):.3f}")
print(f"I(y_true; y_pred) = {I:.3f}")


## 7. Normalized Mutual Information (NMI)

Raw mutual information depends on $H(L_1)$ and $H(L_2)$, so values aren't comparable across datasets. We **normalise** by some combination of the entropies.

The most common choice (and scikit-learn's default) is the **geometric mean** of the entropies:

$$ \text{NMI}(L_1, L_2) \;=\; \frac{I(L_1; L_2)}{\sqrt{H(L_1)\, H(L_2)}} $$

Other variants divide by the arithmetic mean or the maximum of the entropies. All give values in $[0, 1]$ with 1 = perfect agreement and 0 = independence.


In [ ]:
def nmi_from_scratch(y_true, y_pred):
    h_true = entropy(y_true)
    h_pred = entropy(y_pred)
    if h_true == 0 or h_pred == 0:
        return 0.0
    return mutual_information(y_true, y_pred) / np.sqrt(h_true * h_pred)


nmi = nmi_from_scratch(y_true, y_pred)
print(f"NMI from scratch : {nmi:.3f}")

from sklearn.metrics import normalized_mutual_info_score
print(f"NMI from sklearn : {normalized_mutual_info_score(y_true, y_pred):.3f}")


**Both implementations agree.** You've now built both metrics yourself.

## 8. Sanity checks on edge cases

A good way to internalise these scores: see what they give in extreme situations.


In [ ]:
def both_metrics(name, y_true, y_pred):
    ari = adjusted_rand_score(y_true, y_pred)
    nmi = normalized_mutual_info_score(y_true, y_pred)
    print(f"{name:35s}  ARI = {ari:+.3f}   NMI = {nmi:.3f}")


y = np.array([0]*5 + [1]*5 + [2]*5)            # 15 points, 3 classes

both_metrics("perfect prediction",            y, y.copy())
both_metrics("relabelled (same partition)",   y, np.array([2]*5 + [0]*5 + [1]*5))
both_metrics("all points in one cluster",     y, np.zeros_like(y))
both_metrics("each point its own cluster",    y, np.arange(len(y)))
both_metrics("inverse — 'opposite' partition", y, np.array([1,2,0]*5))


In [ ]:
# Random clusterings — on average ARI ≈ 0, NMI > 0
rng = np.random.default_rng(seed=0)
y = np.repeat([0, 1, 2], 30)

aris, nmis = [], []
for _ in range(2000):
    y_rand = rng.integers(0, 3, size=len(y))
    aris.append(adjusted_rand_score(y, y_rand))
    nmis.append(normalized_mutual_info_score(y, y_rand))

print(f"Random clusterings, averaged over 2000 runs:")
print(f"  Mean ARI = {np.mean(aris):+.3f}   (≈ 0, as designed)")
print(f"  Mean NMI = {np.mean(nmis):.3f}    (> 0 — NMI is NOT chance-corrected)")


**Key takeaways from the edge cases:**

- ARI = NMI = 1 when the partitions match (also when only the cluster *names* differ — they're permutation-invariant).
- ARI ≈ 0 for random clusterings. NMI is small but positive — that's why your thesis should report **both**.
- Putting everything into one cluster, or each point into its own cluster, gives ARI = 0 and NMI = 0 — both metrics correctly say „no information".


## 9. How to use them in your thesis

A small recipe — what you'll actually write in your benchmark code:

```python
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score

# 1. Get embeddings from your encoder
Z = encoder(X).detach().cpu().numpy()

# 2. Cluster the embeddings
k = len(np.unique(y_true))
y_pred = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(Z)

# 3. Score against the ground truth
ari   = adjusted_rand_score(y_true, y_pred)
nmi   = normalized_mutual_info_score(y_true, y_pred)
sil   = silhouette_score(Z, y_pred)   # internal — doesn't need ground truth

print(f"NMI = {nmi:.3f}   ARI = {ari:.3f}   Silhouette = {sil:.3f}")
```

A few practical points:

- **Report both** ARI and NMI. They measure slightly different things, and the literature on deep clustering reports both.
- **Average over multiple seeds.** k-means is initialisation-sensitive — run with `n_init=10` (or more), and average your final scores over several training seeds of the encoder.
- **Silhouette** is an *internal* measure (no ground truth needed). For sanity checking embeddings during development it's very useful.
- **Be careful with k.** Both ARI and NMI use the *number of clusters* in the prediction. For your thesis you'll typically set `k = number of true classes`. If you ever set k differently, document it clearly.

That's the whole story of NMI and ARI. You now know exactly what numbers are being reported, why they're trustworthy, and how to compute them by hand if you ever need to debug a suspicious result.
